In [ ]:
!pip install nflreadpy --quiet

import nflreadpy as nfl
import pandas as pd

print("nflreadpy ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.4 MB/s eta 0:00:00
nflreadpy ready


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving FantasyPros_Fantasy_Football_Projections_QB.csv to FantasyPros_Fantasy_Football_Projections_QB.csv
Saving FantasyPros_Fantasy_Football_Projections_RB.csv to FantasyPros_Fantasy_Football_Projections_RB.csv
Saving FantasyPros_Fantasy_Football_Projections_TE.csv to FantasyPros_Fantasy_Football_Projections_TE.csv
Saving FantasyPros_Fantasy_Football_Projections_WR.csv to FantasyPros_Fantasy_Football_Projections_WR.csv


In [ ]:
qb = pd.read_csv('FantasyPros_Fantasy_Football_Projections_QB.csv', skiprows=[1])
rb = pd.read_csv('FantasyPros_Fantasy_Football_Projections_RB.csv', skiprows=[1])
wr = pd.read_csv('FantasyPros_Fantasy_Football_Projections_WR.csv', skiprows=[1])
te = pd.read_csv('FantasyPros_Fantasy_Football_Projections_TE.csv', skiprows=[1])

rb = rb.rename(columns={'ATT':'RUSH_ATT','YDS':'RUSH_YDS','TDS':'RUSH_TDS','YDS.1':'REC_YDS','TDS.1':'REC_TDS'})
wr = wr.rename(columns={'YDS':'REC_YDS','TDS':'REC_TDS','ATT':'RUSH_ATT','YDS.1':'RUSH_YDS','TDS.1':'RUSH_TDS'})

qb['POS'] = 'QB'
rb['POS'] = 'RB'
wr['POS'] = 'WR'
te['POS'] = 'TE'

combined = pd.concat([qb, rb, wr, te], ignore_index=True)
combined = combined.dropna(subset=['Player'])
combined['FPTS'] = pd.to_numeric(combined['FPTS'], errors='coerce')
combined = combined.dropna(subset=['FPTS'])

replacement = {
    'QB': combined[combined.POS=='QB'].sort_values('FPTS', ascending=False).iloc[19]['FPTS'],
    'RB': combined[combined.POS=='RB'].sort_values('FPTS', ascending=False).iloc[30]['FPTS'],
    'WR': combined[combined.POS=='WR'].sort_values('FPTS', ascending=False).iloc[30]['FPTS'],
    'TE': combined[combined.POS=='TE'].sort_values('FPTS', ascending=False).iloc[13]['FPTS'],
}
combined['VORP'] = combined.apply(lambda r: round(r['FPTS'] - replacement[r['POS']], 1), axis=1)

print(combined.shape)

(526, 20)


In [ ]:
top250 = combined.sort_values('VORP', ascending=False).head(250).reset_index(drop=True)

top250[['Player', 'Team', 'POS', 'FPTS', 'VORP']].head(10)

,Player,Team,POS,FPTS,VORP
0,Jahmyr Gibbs,DET,RB,372.6,186.7
1,Bijan Robinson,ATL,RB,372.0,186.1
2,Christian McCaffrey,SF,RB,334.8,148.9
3,Puka Nacua,LAR,WR,339.8,142.5
4,Ja'Marr Chase,CIN,WR,336.1,138.8
5,Jaxon Smith-Njigba,SEA,WR,324.0,126.7
6,Jonathan Taylor,IND,RB,312.4,126.5
7,Amon-Ra St. Brown,DET,WR,319.6,122.3
8,De'Von Achane,MIA,RB,292.0,106.1
9,Josh Allen,BUF,QB,372.2,97.7


In [ ]:
weekly = nfl.load_player_stats(seasons=[2025])
weekly = weekly.to_pandas()

print(weekly.shape)
weekly.head()

(19421, 145)


,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,...,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
0,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2025,1,REG,2025_01_PIT_NYJ,...,0,0,0,0,0,0,0,0,25.66,25.66
1,00-0023853,M.Prater,Matt Prater,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,2025,1,REG,2025_01_BAL_BUF,...,0,0,0,0,0,0,0,0,0.00,0.00
2,00-0025565,N.Folk,Nick Folk,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,2025,1,REG,2025_01_PIT_NYJ,...,0,0,0,0,0,0,0,0,0.00,0.00
3,00-0026158,J.Flacco,Joe Flacco,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2025,1,REG,2025_01_CIN_CLE,...,0,0,0,0,0,0,0,0,12.20,12.20
4,00-0026190,C.Campbell,Calais Campbell,DE,DL,https://static.www.nfl.com/image/upload/f_auto...,2025,1,REG,2025_01_ARI_NO,...,0,0,0,0,0,0,0,0,0.00,0.00


In [ ]:
weekly_top250 = weekly[weekly['player_display_name'].isin(top250['Player'])]

weekly_top250 = weekly_top250[weekly_top250['season_type'] == 'REG']

print(weekly_top250.shape)
print(weekly_top250['player_display_name'].nunique(), "unique players matched")

(2981, 145)
213 unique players matched


In [ ]:
matched_names = weekly_top250['player_display_name'].unique()
missing = top250[~top250['Player'].isin(matched_names)]

missing[['Player', 'Team', 'POS', 'VORP']]

,Player,Team,POS,VORP
15,James Cook III,BUF,RB,84.3
23,Jeremiyah Love,ARI,RB,69.4
38,Travis Etienne Jr.,NO,RB,46.1
49,Kyle Pitts Sr.,ATL,TE,32.2
54,Patrick Mahomes II,KC,QB,29.1
88,Jadarian Price,SEA,RB,3.7
101,Michael Pittman Jr.,PIT,WR,-7.3
112,Carnell Tate,TEN,WR,-15.9
115,Aaron Jones Sr.,MIN,RB,-17.8
125,Jordyn Tyson,NO,WR,-24.8


In [ ]:
import re

def clean_name(name):
    name = re.sub(r'\s+(Jr\.?|Sr\.?|II|III|IV)$', '', name)
    return name.strip()

top250['clean_name'] = top250['Player'].apply(clean_name)
weekly['clean_name'] = weekly['player_display_name'].apply(clean_name)

weekly_top250 = weekly[weekly['clean_name'].isin(top250['clean_name'])]
weekly_top250 = weekly_top250[weekly_top250['season_type'] == 'REG']

print(weekly_top250.shape)
print(weekly_top250['clean_name'].nunique(), "unique players matched")

TypeError: expected string or bytes-like object, got 'NoneType'

In [ ]:
def clean_name(name):
    if not isinstance(name, str):
        return name
    name = re.sub(r'\s+(Jr\.?|Sr\.?|II|III|IV)$', '', name)
    return name.strip()

top250['clean_name'] = top250['Player'].apply(clean_name)
weekly['clean_name'] = weekly['player_display_name'].apply(clean_name)

weekly_top250 = weekly[weekly['clean_name'].isin(top250['clean_name'])]
weekly_top250 = weekly_top250[weekly_top250['season_type'] == 'REG']

print(weekly_top250.shape)
print(weekly_top250['clean_name'].nunique(), "unique players matched")

(3092, 146)
220 unique players matched


In [ ]:
matched_names = weekly_top250['clean_name'].unique()
still_missing = top250[~top250['clean_name'].isin(matched_names)]

still_missing[['Player', 'Team', 'POS', 'VORP']]

,Player,Team,POS,VORP
23,Jeremiyah Love,ARI,RB,69.4
88,Jadarian Price,SEA,RB,3.7
112,Carnell Tate,TEN,WR,-15.9
125,Jordyn Tyson,NO,WR,-24.8
134,Kenyon Sadiq,NYJ,TE,-33.3
137,Makai Lemon,PHI,WR,-34.9
144,Jonathon Brooks,CAR,RB,-41.3
161,KC Concepcion,CLE,WR,-54.6
164,Tank Dell,HOU,WR,-56.0
174,De'Zhaun Stribling,SF,WR,-68.4


In [ ]:
player_weekly_stats = weekly_top250.groupby('clean_name')['fantasy_points_ppr'].agg(
    games_played='count',
    avg_2025='mean',
    p10=lambda x: x.quantile(0.10),
    p90=lambda x: x.quantile(0.90),
).reset_index()

print(player_weekly_stats.shape)
player_weekly_stats.sort_values('avg_2025', ascending=False).head(10)

(220, 5)


,clean_name,games_played,avg_2025,p10,p90
42,Christian McCaffrey,17,24.505882,14.100,34.620
167,Puka Nacua,16,23.437500,13.750,35.850
125,Josh Allen,16,22.788750,9.970,38.300
15,Bijan Robinson,17,21.811765,8.560,32.500
97,Jahmyr Gibbs,17,21.582353,7.060,37.480
121,Jonathan Taylor,17,21.311765,8.240,35.480
111,Jaxon Smith-Njigba,17,21.170588,13.840,28.940
72,Drake Maye,17,20.703529,15.620,26.676
153,Matthew Stafford,17,20.610588,12.304,27.388
166,Patrick Mahomes,14,20.405714,11.268,28.798


In [ ]:
top250_stats = top250.merge(player_weekly_stats, on='clean_name', how='left')

top250_stats['proj_avg_per_game'] = top250_stats['FPTS'] / 17

top250_stats['floor_ratio'] = top250_stats['p10'] / top250_stats['avg_2025']
top250_stats['ceiling_ratio'] = top250_stats['p90'] / top250_stats['avg_2025']

top250_stats['proj_floor'] = (top250_stats['proj_avg_per_game'] * top250_stats['floor_ratio']).round(1)
top250_stats['proj_ceiling'] = (top250_stats['proj_avg_per_game'] * top250_stats['ceiling_ratio']).round(1)

top250_stats[['Player', 'POS', 'FPTS', 'proj_avg_per_game', 'proj_floor', 'proj_ceiling']].sort_values('VORP', ascending=False).head(10)

KeyError: 'VORP'

In [ ]:
top250_stats = top250.merge(player_weekly_stats, on='clean_name', how='left')

top250_stats['proj_avg_per_game'] = top250_stats['FPTS'] / 17

top250_stats['floor_ratio'] = top250_stats['p10'] / top250_stats['avg_2025']
top250_stats['ceiling_ratio'] = top250_stats['p90'] / top250_stats['avg_2025']

top250_stats['proj_floor'] = (top250_stats['proj_avg_per_game'] * top250_stats['floor_ratio']).round(1)
top250_stats['proj_ceiling'] = (top250_stats['proj_avg_per_game'] * top250_stats['ceiling_ratio']).round(1)

top250_stats[['Player', 'POS', 'FPTS', 'proj_avg_per_game', 'proj_floor', 'proj_ceiling']].sort_values('FPTS', ascending=False).head(10)

,Player,POS,FPTS,proj_avg_per_game,proj_floor,proj_ceiling
0,Jahmyr Gibbs,RB,372.6,21.917647,7.2,38.1
9,Josh Allen,QB,372.2,21.894118,9.6,36.8
1,Bijan Robinson,RB,372.0,21.882353,8.6,32.6
3,Puka Nacua,WR,339.8,19.988235,11.7,30.6
4,Ja'Marr Chase,WR,336.1,19.770588,6.7,33.0
2,Christian McCaffrey,RB,334.8,19.694118,11.3,27.8
31,Drake Maye,QB,326.9,19.229412,14.5,24.8
33,Jayden Daniels,QB,325.5,19.147059,12.0,24.3
34,Lamar Jackson,QB,325.0,19.117647,5.9,31.1
5,Jaxon Smith-Njigba,WR,324.0,19.058824,12.5,26.1


In [ ]:
position_avg_ratios = top250_stats.groupby('POS')[['floor_ratio', 'ceiling_ratio']].mean().reset_index()
position_avg_ratios.columns = ['POS', 'pos_floor_ratio', 'pos_ceiling_ratio']

top250_stats = top250_stats.merge(position_avg_ratios, on='POS', how='left')

no_history = top250_stats['avg_2025'].isna()
top250_stats.loc[no_history, 'proj_floor'] = (top250_stats.loc[no_history, 'proj_avg_per_game'] * top250_stats.loc[no_history, 'pos_floor_ratio']).round(1)
top250_stats.loc[no_history, 'proj_ceiling'] = (top250_stats.loc[no_history, 'proj_avg_per_game'] * top250_stats.loc[no_history, 'pos_ceiling_ratio']).round(1)

top250_stats[top250_stats['clean_name'].isin(missing['Player'].apply(clean_name))][['Player','POS','FPTS','proj_avg_per_game','proj_floor','proj_ceiling']]

,Player,POS,FPTS,proj_avg_per_game,proj_floor,proj_ceiling
15,James Cook III,RB,270.2,15.894118,6.4,25.3
23,Jeremiyah Love,RB,255.3,15.017647,5.4,26.3
38,Travis Etienne Jr.,RB,232.0,13.647059,5.4,19.0
49,Kyle Pitts Sr.,TE,195.6,11.505882,3.8,17.0
54,Patrick Mahomes II,QB,303.6,17.858824,9.9,25.2
88,Jadarian Price,RB,189.6,11.152941,4.0,19.6
101,Michael Pittman Jr.,WR,190.0,11.176471,3.2,19.4
112,Carnell Tate,WR,181.4,10.670588,3.2,19.3
115,Aaron Jones Sr.,RB,168.1,9.888235,4.5,15.7
125,Jordyn Tyson,WR,172.5,10.147059,3.1,18.4


In [ ]:
still_null = top250_stats['proj_floor'].isna().sum()
print(still_null, "players still missing floor/ceiling")

top250_stats[['Player','POS','FPTS','proj_floor','proj_ceiling']].to_csv('floor_ceiling_2026.csv', index=False)

from google.colab import files
files.download('floor_ceiling_2026.csv')

0 players still missing floor/ceiling


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>